In [2]:
import torch
import torch.nn.functional as F

In [3]:
words = open('../names.txt','r').read().splitlines()
chs = sorted(list(set(''.join(words))))
stoi = {c:i+1 for i,c in enumerate(chs)}
stoi['.'] = 0
itos = {stoi[c]:c for c in stoi}


In [4]:
N = torch.zeros(27,27)
for word in words:
    pW = '.' + word + '.'
    for ch1, ch2 in zip(pW, pW[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        N[ix1][ix2]+=1


In [25]:
N += 1
N = N.float()
P = N/N.sum(dim=1, keepdim=True)
N.sum(dim=1, keepdim=True)
N.shape

torch.Size([27, 27])

In [12]:
g = torch.Generator().manual_seed(2147483647)
for i in range(5):
    ix = 0
    word = []
    while True:
        ix = torch.multinomial(P[ix], num_samples = 1, generator = g).item()
        if ix==0:
            break
        ch = itos[ix]
        word.append(ch)
    print("".join(word))
        



cexze
momasurailezitynn
konimittain
llayn
ka


In [7]:
logP = torch.log(P)
logSum = 0
totalPairs = 0
for word in words:
     pW = '.' + word + '.'
     for ch1, ch2 in zip(pW, pW[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        logSum+=logP[ix1, ix2]
        totalPairs+=1
loss = -logSum/totalPairs
loss.item()

2.4543561935424805

In [8]:
xs = []
ys = []
total = 0
for word in words:
    pW = '.' + word + '.'
    for ch1, ch2 in zip(pW, pW[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        xs.append(ix1)
        ys.append(ix2)
        total+=1

In [9]:
v = F.one_hot(torch.tensor(xs), num_classes=27).float()
g = torch.Generator().manual_seed(2147483647)
W = torch.randn(27,27, generator=g, requires_grad=True)

In [ ]:
for i in range(100):

    logits = v @ W
    counts = logits.exp()
    probs = counts/counts.sum(1, keepdim=True)
    logProb = torch.log(probs[torch.arange(total), ys])
    loss = -torch.mean(logProb) + 0.01 * (W**2).mean()

    W.grad = None

    
    loss.backward()

    W.data -= 50 * W.grad

    



tensor([-3.0441, -3.2863, -3.7473,  ..., -4.8359, -5.0043, -1.5700],
       grad_fn=<LogBackward0>)
tensor([-3.0441, -3.2863, -3.7470,  ..., -4.8356, -5.0077, -1.5697],
       grad_fn=<LogBackward0>)
tensor([-3.0441, -3.2863, -3.7467,  ..., -4.8352, -5.0111, -1.5694],
       grad_fn=<LogBackward0>)
tensor([-3.0441, -3.2863, -3.7464,  ..., -4.8349, -5.0145, -1.5691],
       grad_fn=<LogBackward0>)
tensor([-3.0441, -3.2863, -3.7461,  ..., -4.8346, -5.0178, -1.5688],
       grad_fn=<LogBackward0>)
tensor([-3.0441, -3.2863, -3.7458,  ..., -4.8342, -5.0212, -1.5685],
       grad_fn=<LogBackward0>)
tensor([-3.0441, -3.2863, -3.7455,  ..., -4.8339, -5.0245, -1.5683],
       grad_fn=<LogBackward0>)
tensor([-3.0441, -3.2863, -3.7452,  ..., -4.8336, -5.0277, -1.5680],
       grad_fn=<LogBackward0>)
tensor([-3.0441, -3.2863, -3.7449,  ..., -4.8333, -5.0310, -1.5677],
       grad_fn=<LogBackward0>)
tensor([-3.0441, -3.2863, -3.7447,  ..., -4.8330, -5.0342, -1.5675],
       grad_fn=<LogBackward0>)


In [11]:
g = torch.Generator().manual_seed(2147483647)
for i in range(5):
    ix = 0
    word = []
    while True:
        xenc = F.one_hot(torch.tensor([ix]), num_classes=27).float()
        logits = xenc @ W
        counts = logits.exp()
        probs = counts/counts.sum(1, keepdim=True)
        ix = torch.multinomial(probs, num_samples = 1, generator = g).item()
        word.append(itos[ix])
        if ix==0:
            break
    print("".join(word))
        

cexze.
momasurailezityha.
konimittain.
llayn.
ka.
